In [ ]:
%cd zarr-haploatlas/prep/genome-annotations

In [1]:
import pandas as pd

df = pd.read_csv(
    'PlasmoDB-54_Pfalciparum3D7.gff',
    sep='\t',
    comment='#',
    header=None,
    names=['seqid', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes']
)

# Parse attributes into separate columns
def parse_attributes(attr_string):
    """Parse GFF3 attributes into a dictionary"""
    attrs = {}
    for item in attr_string.split(';'):
        if '=' in item:
            key, value = item.split('=', 1)
            attrs[key] = value
    return attrs

# Apply to each row
attr_df = df['attributes'].apply(parse_attributes).apply(pd.Series)

# Combine with original dataframe
df = pd.concat([df, attr_df], axis=1).drop(columns = ["source", "attributes", "Note", "score", "protein_source_id"]).sort_values("gene_id")

In [2]:
gene_desc = df.loc[~df.Name.isna() | ~df.description.isna()].copy()

gene_desc["description"] = gene_desc.fillna("").apply(lambda x: x["description"] if len(x["Name"]) == 0 else x["description"] + " (" + x["Name"] + ")", axis = 1)
gene_desc = gene_desc.loc[gene_desc.ID.str.split(".").str.len() == 1].sort_values("ID")

gene_desc_map = dict(zip(gene_desc.ID, gene_desc.description))

filtered = df.loc[
    ~df["type"].isin(["exon", "mRNA", "protein_coding_gene", "pseudogenic_transcript", "ncRNA_gene", "pseudogene"])
].copy()

filtered.Parent  = filtered.Parent.str.split(".").str[0]
filtered.loc[filtered.ID.str.contains("CDS"), "exon"] = filtered.loc[filtered.ID.str.contains("CDS"), "ID"].str.partition("CDS")[2]
filtered.gene_id = filtered.gene_id.combine_first(filtered.Parent).combine_first(filtered.ID)

filtered = filtered.drop(columns = ["ID", "Parent", "Name"]).sort_values(["gene_id", "exon"]).reset_index(drop = True)
filtered.loc[filtered.description.isna(), "description"] = filtered.loc[filtered.description.isna(), "gene_id"].map(gene_desc_map)


def assign_tracks(df, gap):
    df = df.sort_values(["chrom", "min_start"])
    track_ends = {}  # chrom -> list of end positions per track
    y = []

    for _, row in df.iterrows():
        chrom = row["chrom"]
        if chrom not in track_ends:
            track_ends[chrom] = []
        
        assigned = None
        for i, end in enumerate(track_ends[chrom]):
            if row["min_start"] > end + gap:
                assigned = i
                track_ends[chrom][i] = row["max_end"]
                break

        if assigned is None:
            assigned = len(track_ends[chrom])
            track_ends[chrom].append(row["max_end"])

        y.append(assigned)

    df = df.copy()
    df["y_track"] = y
    return df.reset_index()

y_axis_allocation = filtered.groupby("gene_id").apply(lambda x:
    pd.Series({"chrom": x.seqid.values[0], "min_start": x.start.min(), "max_end": x.end.max()}), include_groups = False)

y_axis_allocation = assign_tracks(y_axis_allocation, gap = 50)

final = filtered.merge(y_axis_allocation[["gene_id", "y_track"]], on = "gene_id").drop_duplicates().reset_index(drop = True)

In [3]:
final

,seqid,type,start,end,strand,phase,description,gene_id,exon,y_track
0,Pf3D7_01_v3,CDS,29510,34762,+,0,erythrocyte membrane protein 1%2C PfEMP1 (VAR),PF3D7_0100100,1,0
1,Pf3D7_01_v3,CDS,35888,37126,+,0,erythrocyte membrane protein 1%2C PfEMP1 (VAR),PF3D7_0100100,2,0
2,Pf3D7_01_v3,CDS,40154,40207,-,0,rifin (RIF),PF3D7_0100200,1,0
3,Pf3D7_01_v3,CDS,38982,39923,-,0,rifin (RIF),PF3D7_0100200,2,0
4,Pf3D7_01_v3,CDS,43775,46507,-,0,erythrocyte membrane protein 1%2C PfEMP1 (VAR),PF3D7_0100300,1,0
...,...,...,...,...,...,...,...,...,...,...
23689,Pf3D7_MIT_v3,rRNA,5508,5546,-,.,ribosomal RNA fragment RNA14,PF3D7_MIT03800,NaN,1
23690,Pf3D7_MIT_v3,rRNA,5547,5576,-,.,ribosomal RNA fragment RNA19,PF3D7_MIT03900,NaN,2
23691,Pf3D7_MIT_v3,rRNA,5577,5771,-,.,large subunit ribosomal RNA fragment E,PF3D7_MIT04000,NaN,0
23692,Pf3D7_MIT_v3,rRNA,5772,5854,-,.,large subunit ribosomal RNA fragment D,PF3D7_MIT04100,NaN,1


In [23]:
from bokeh.plotting import figure, show
from bokeh.models import (ColumnDataSource, HoverTool, BasicTicker, FixedTicker,
                          NumeralTickFormatter, Range1d, BoxAnnotation,
                          LabelSet, CustomJS, Grid, LinearAxis)
from bokeh.io import output_notebook

proteins = {}
with open("PlasmoDB-68_Pfalciparum3D7_AnnotatedProteins.fasta") as f:
    gene_id = None
    seq = []
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            if gene_id:
                proteins[gene_id] = "".join(seq)
            gene_id = None
            seq = []
            for field in line.split("|"):
                field = field.strip()
                if field.startswith("gene="):
                    gene_id = field.split("=")[1]
        else:
            seq.append(line)
    if gene_id:
        proteins[gene_id] = "".join(seq)

codon_records = []


def build_codon_df(cds_df):
    for gene_id, grp in cds_df.groupby("gene_id"):
        strand = grp["strand"].values[0]
        y_track = grp["y_track"].values[0]
    
        # Sort exons in reading order
        if strand == "+":
            exons = grp.sort_values("start")
        else:
            exons = grp.sort_values("end", ascending=False)
    
        # Collect genomic base positions in reading order
        positions = []
        for _, exon in exons.iterrows():
            if strand == "+":
                positions.extend(range(exon["start"], exon["end"] + 1))
            else:
                positions.extend(range(exon["end"], exon["start"] - 1, -1))
    
        # Skip bases according to phase of first CDS in reading order
        first_phase = exons.iloc[0]["phase"]
        first_phase = int(first_phase) if first_phase != "." else 0
        positions = positions[first_phase:]
    
        protein = proteins.get(gene_id, "")
    
        # Create one bar per codon
        for i in range(0, len(positions) - 2, 3):
            codon_pos = positions[i:i+3]
            if len(codon_pos) < 3:
                break
            aa_idx = i // 3
            aa = protein[aa_idx] if aa_idx < len(protein) else "*"
    
            codon_records.append({
                "left": min(codon_pos) - 0.5,
                "right": max(codon_pos) + 0.5,  # each base P occupies [P-0.5, P+0.5]
                "y_track": y_track,
                "aa": aa,
                "aa_pos": aa_idx + 1,
                "gene_id": gene_id,
            })
    
    return pd.DataFrame(codon_records)
    
def main(locus, annotations):
    chromosome = locus.split(":")[0]
    coords     = locus.split(":")[1]
    start      = int(coords.split("-")[0].replace(",", ""))
    end        = int(coords.split("-")[1].replace(",", ""))
    # Consider setting a maximum zoom out distance to reduce rendering times
    
    span = end - start
    start_initial = start - span * 3 - 20
    end_initial   = end + span * 3 + 20

    color_map = {
        "five_prime_UTR": "#008000",
        "three_prime_UTR": "#FF0000",
        "CDS": "#4CAF50",
    }
    annotations["color"] = annotations["type"].map(color_map).fillna("#999999")

    subset   = annotations.loc[annotations.seqid == chromosome]
    non_cds  = subset.loc[subset["type"] != "CDS"]
    cds_only = subset.loc[subset["type"] == "CDS"]

    codon_df = build_codon_df(cds_only)

    output_notebook()
    p = figure(
        width=1200, height=300,
        title="Gene Tracks",
        x_axis_label="Genomic Position",
        tools="xpan,xwheel_zoom,reset",
        active_scroll="xwheel_zoom",
        output_backend="webgl",  # GPU-accelerated quads
    )

    p.yaxis.visible = False
    p.ygrid.grid_line_color = None
    p.xaxis.ticker = BasicTicker(min_interval=1)
    p.xaxis.formatter = NumeralTickFormatter(format="0,0")
    p.xaxis.major_tick_line_color = None
    p.xaxis.major_tick_in = 0
    p.xaxis.major_tick_out = 0
    p.xaxis.minor_tick_line_color = None
    p.xgrid.grid_line_color = None
    p.xgrid.minor_grid_line_color = None
    p.x_range = Range1d(
        start=start_initial, end=end_initial,
        bounds=(non_cds["start"].min(), non_cds["end"].max()),
        min_interval=5,
    )

    # ── Boundary gridlines (visible only when ≤5 integers on screen) ──
    boundary_ticker = FixedTicker(ticks=[])
    boundary_grid = Grid(
        dimension=0, ticker=boundary_ticker,
        grid_line_color="#cccccc", grid_line_alpha=0.8,
    )
    p.add_layout(boundary_grid)
    boundary_axis = LinearAxis(ticker=boundary_ticker)
    boundary_axis.major_label_text_font_size = "0pt"
    boundary_axis.major_label_text_color = None
    boundary_axis.major_tick_line_color = "#666666"
    boundary_axis.major_tick_in = 0
    boundary_axis.major_tick_out = 4
    boundary_axis.minor_tick_line_color = None
    boundary_axis.axis_line_color = None
    p.add_layout(boundary_axis, "below")

    # --- Gene-level bars (few enough to render all) ---
    gene_source = ColumnDataSource(dict(
        left=(non_cds["start"] - 0.5).values, right=(non_cds["end"] + 0.5).values,
        top=(non_cds["y_track"] + 0.4).values, bottom=(non_cds["y_track"] - 0.4).values,
        gene_id=non_cds["gene_id"].values, strand=non_cds["strand"].values,
        desc=non_cds["description"].values, feat_type=non_cds["type"].values,
        color=non_cds["color"].values,
    ))
    p.quad(left="left", right="right", top="top", bottom="bottom",
           source=gene_source, color="color", alpha=0.8, line_width=0)
    p.add_tools(HoverTool(tooltips=[
        ("Gene", "@gene_id"), ("Type", "@feat_type"), ("Strand", "@strand"),
        ("Position", "@left{0,0} - @right{0,0}"), ("Description", "@desc"),
    ]))

    # ROI
    p.add_layout(BoxAnnotation(left=start, right=end,
                               fill_alpha=0.15, fill_color="orange", line_width=0))

    # --- Exon bars (few enough to render all) ---
    exon_source = ColumnDataSource(dict(
        left=(cds_only["start"] - 0.5).values, right=(cds_only["end"] + 0.5).values,
        top=(cds_only["y_track"] + 0.4).values, bottom=cds_only["y_track"].values,
        gene_id=cds_only["gene_id"].values,
    ))
    p.quad(left="left", right="right", top="top", bottom="bottom",
           source=exon_source, color="grey", alpha=0.5)

    # --- Codon bars: FULL data (hidden) + RENDERED data (visible window only) ---
    full_codon = ColumnDataSource(dict(
        left=codon_df["left"].values.astype(float),
        right=codon_df["right"].values.astype(float),
        top=codon_df["y_track"].values.astype(float),
        bottom=(codon_df["y_track"] - 0.4).values.astype(float),
        aa=codon_df["aa"].values,
        aa_pos=codon_df["aa_pos"].astype(str).values,
        gene_id=codon_df["gene_id"].values,
        x_mid=((codon_df["left"] + codon_df["right"]) / 2).values.astype(float),
        y_mid=(codon_df["y_track"] - 0.2).values.astype(float),
        text=(codon_df["aa"] + codon_df["aa_pos"].astype(str)).values,
    ))

    rendered_codon = ColumnDataSource(dict(
        left=[], right=[], top=[], bottom=[], aa=[], aa_pos=[], gene_id=[],
    ))
    rendered_labels = ColumnDataSource(dict(x=[], y=[], text=[]))

    p.quad(left="left", right="right", top="top", bottom="bottom",
           source=rendered_codon, color="lightblue", alpha=0.8,
           line_color="blue", line_width=0.8)

    labels = LabelSet(x="x", y="y", text="text", source=rendered_labels,
                       text_align="center", text_baseline="middle", text_font_size="6pt")
    p.add_layout(labels)

    # --- CustomJS: filter codons/labels to visible window on every pan/zoom ---
    callback = CustomJS(args=dict(
        full=full_codon, rendered=rendered_codon,
        rendered_labels=rendered_labels, x_range=p.x_range,
        boundary_ticker=boundary_ticker,
    ), code="""
        const lo = x_range.start;
        const hi = x_range.end;
        const span = hi - lo;

        // ── Boundary gridlines at half-integers (between tick labels) ──
        // Only visible when zoomed to ≤50 integers on screen
        if (span > 51) {
            boundary_ticker.ticks = [];
        } else {
            const ticks = [];
            const first = Math.ceil(lo - 0.5);
            for (let k = first; k <= hi + 0.5; k++) {
                const v = k + 0.5;
                if (v >= lo && v <= hi) ticks.push(v);
            }
            boundary_ticker.ticks = ticks;
        }

        // ── Codon bars ──
        const f = full.data;
        const n = f['left'].length;
        const keys = ['left','right','top','bottom','aa','aa_pos','gene_id'];
        const out = {};
        for (const k of keys) out[k] = [];

        for (let i = 0; i < n; i++) {
            if (f['right'][i] >= lo && f['left'][i] <= hi) {
                for (const k of keys) out[k].push(f[k][i]);
            }
        }
        rendered.data = out;

        // ── Labels: only when zoomed in enough ──
        const lout = {x: [], y: [], text: []};
        if (span < 200) {
            for (let i = 0; i < n; i++) {
                if (f['right'][i] >= lo && f['left'][i] <= hi) {
                    lout.x.push(f['x_mid'][i]);
                    lout.y.push(f['y_mid'][i]);
                    lout.text.push(f['text'][i]);
                }
            }
        }
        rendered_labels.data = lout;
    """)

    p.x_range.js_on_change('start', callback)
    p.x_range.js_on_change('end', callback)

    show(p)

In [ ]:
main("Pf3D7_07_v3:403,220-403,300", final); # semicolon is important to prevent some strange bokeh-jupyter rendering artefact

Loading BokehJS ...